# 4 Model IO
## 3.2 Prompt Templates
3. FewShotPromptTemplate 
    - 모델에 대답할 형식을 예제로 준다. 데이터베이스에 예제가 있을 경우 사용.

4. FewShotChatMessagePromptTemplate
    - 채팅 메시지를 위한 템플릿

In [ ]:
# 3. Few Shot Prompt Template

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

# examples fetching from database
examples = [ 
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# 예제 형식 지정
example_template = """ 
    Human: {question}
    AI: {answer}
""" 

# format example
example_prompt = PromptTemplate.from_template(example_template) # == PromptTemplate.from_template("Human: {question}\nAI:{answer}")

# fewshot을 이용한 실제 prompt 만들기
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix= "Human: What do you know about {country}?", # question of a user.
    input_variables=["country"] # for validation
)

prompt.format(country="Germany")

chain = prompt | chat
chain.invoke({
    "country":"Germany"
})

In [ ]:
# 4. FewShotChatMessagePromptTemplate

from langchain.chat_models import ChatOpenAI
from langchain.prompts import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import ChatPromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# How to format examples
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about {country}?"),
    ("ai", "{answer}"),
])

# 예제 목록에 있는 각 예제에 형식을 지정한다.
prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. You give short answers."),
    prompt,
    ("human", "What do you know about {country}?")
])

chain = final_prompt | chat
chain.invoke({"country": "Germany"})

## 4.3 Example Selectors
1. LengthBasedExampleSelector VS 4.1 FewShotPromptTemplate
    - FewShot은 모든 예제를 format하는 반면, ExampleSelector는 설정값에 따라 prompt에 알맞은 예제를 골라준다.
    - format examples and see how long they are.

2. Custom Example Selector

In [ ]:
# 1. LengthBasedExampleSelector

from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate, LengthBasedExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=180,
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")

In [ ]:
# 2. Custom Example Selector

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector.base import BaseExampleSelector
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# Select one random example from examples.
class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples: list):
        self.examples = examples

    # examples list에 example 추가하는 method.
    def add_example(self, example):
        self.examples.append(example)
    
    def select_examples(self, input_variables) -> list[dict]:
        from random import choice
        return [choice(self.examples)]

example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")

# 4.4 Serialization(직렬화)
    - disk에서 prompt template을 저장하고 불러오기
    - two types of prompts
        1. JSON
        2. YAML

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import load_prompt

chat = ChatOpenAI()

prompt = load_prompt("./prompt.yaml")
prompt.format(country="Germany")

'What is the capital of Germany'

# 4.4 Composition
    - 작은 prompt template들을 결합하는 방법

In [ ]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts.pipeline import PipelinePromptTemplate

chat = ChatOpenAI()

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
    """
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
    """
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
    """
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
    """
)

prompts = [
    ("intro",intro),
    ("example",example),
    ("start",start),
]

full_prompt = PipelinePromptTemplate(
    final_prompt=final, 
    pipeline_prompts=prompts,
)

full_prompt.format(
    character = "Remi",
    example_question = "Which show are you in?",
    example_answer = "Doremi",
    question = "Who are your pioneers?",
)

chain = full_prompt | chat
chain.invoke({
    "character": "Remi",
    "example_question": "Which show are you in?",
    "example_answer": "Doremi",
    "question": "Who are your pioneers?",
})

# 4.5 [Caching](https://python.langchain.com/docs/integrations/llm_caching/)
    - Language Model의 Response를 저장할 수 있다.
    - 챗봇이 자주 받는 질문의 답변을 저장하여 재사용

In [ ]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

set_llm_cache(InMemoryCache()) # 모든 response를 메모리에 저장. 노트북 재부팅 시 캐시가 사라짐.
set_llm_cache(SQLiteCache(".langchain.db")) # 데이터베이스에 캐시 저장. 파일을 생성해 줌.
set_debug(True) # Show Log

chat = ChatOpenAI(temperature=0.1)

chat.predict("How do you make Woolong Milk Tea?")

# 4.6 Serialization
1. 비용 예측
2. 모델 저장과 불러오기

In [ ]:
# 1. 비용 예측

from langchain.chat_models.openai import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(
    temperature=0.1,
)

with get_openai_callback() as usage:
    chat.predict("How to make Banh Mi sandwich and the bread")
    # with문부터 print 사이의 모든 코드에 대한 사용량 및 비용이 출력된다.
    print(usage)
    print(usage.total_cost) # 비용만 보기

In [ ]:
# 2-1. 모델 저장
from langchain.llms.openai import OpenAI

chat = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model="gpt-4o-mini",
)

chat.save("model.json")

# 2-2. 모델 불러오기
from langchain.llms.loading import load_llm

chat = load_llm("model.json")
chat